# **Fine-tuning phase**

While my previous script (inference model.py) used a pre-existing model to answer questions, this code actually trains a model to learn the specific patterns and terminology of Austrian tax law using the Unsloth library, which is significantly faster and uses less memory than standard methods.

**Phase 1: Setup and Model Loading**

By installing the engine and loading a "base" model that is already compressed to fit on a standard GPU (like the T4 in Google Colab).


In [ ]:
# Install the Unsloth library, which provides 2x faster training and 70% less memory usage!
# pip install "unsloth[colab-new]" "--upgrade" # Install unsloth and its dependencies, optimized for Colab T4 GPUs.
import torch
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Configuration: Set the maximum length for questions/answers (2048 tokens)
max_seq_length = 2048
# I used Llama 3.1 8B, pre-quantized to 4-bit to ensure it doesn't crash the Colab GPU
model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit" # Pre-quantized for memory safety

# Load the model and its tokenizer (the tool that turns words into numbers)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    token = "hugging_face_token", # Hugging Face token
)

**Phase 2: Applying LoRA (Low-Rank Adaptation)**

Instead of training all 8 billion parameters (which is impossible on a single GPU), we attach "adapters." Think of this as adding a small, specialized "tax law module" to a giant brain.

In [ ]:
# 2. Add LoRA Adapters (Uses 30% less VRAM with 'unsloth' patching)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: The 'size' of the update layer; 16 is a balanced value
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized for Unsloth's speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial for preventing crashes; forces the model to discard/recompute data to save VRAM
)

**Phase 3: Dataset Formatting**

AI models need data in a very specific string format to "understand" where a question ends and an answer begins.

In [ ]:
# 3. Data Preparation (Loading your training_data.json)
# # Open the JSON file containing the Austrian Tax questions and answers
with open('training_data.json', 'r', encoding = 'utf-8') as f:
    json_data = json.load(f)

# Helper function to wrap your data in Llama-3's preferred prompt structure
def format_prompts(example):
    # Standard Llama-3 formatting
    return {"text": f"### Frage: {example['question']}\n### Antwort: {example['answer']} <|end_of_text|>"}

# Convert the raw JSON list into a 'Hugging Face Dataset' and apply the formatting
dataset = Dataset.from_list(json_data)
dataset = dataset.map(format_prompts)

**Phase 4: Training Parameters**

This defines the "rules" for the training session, i.e., how fast the model learns and how it manages memory.

In [ ]:
# 4. Training Arguments: Memory Safe Settings
training_args = TrainingArguments(
    per_device_train_batch_size = 2, # Process 2 examples at a time
    gradient_accumulation_steps = 4, # Effectively mimics a batch size of 8 (2*4)
    warmup_steps = 5, # Gradually increase learning rate to avoid "shocking" the model
    max_steps = 60, # Number of training iterations
    learning_rate = 2e-4, # The speed of learning (0.0002)
    fp16 = not torch.cuda.is_bf16_supported(), # Use 16-bit floats for speed
    logging_steps = 1, # Report progress every single step
    optim = "adamw_8bit", # Use an 8-bit version of the optimizer to save memory
    output_dir = "outputs", # Where to save temporary checkpoints
)

**Phase 5: The Actual Training**

This is the "heavy lifting" part where the GPU calculates the math to update the model's weights.

In [ ]:

# 5. Execution
# The Trainer is the 'orchestrator' that brings the model, data, and arguments together
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
)

# Start the training process
trainer.train()

**Phase 6: Saving the Result**

After training, I saved only the "adapters" (the tax-specific knowledge), which is much smaller than the full model file.

In [ ]:
# 6. Save the LoRA weights and the tokenizer so I can load them later for inference
model.save_pretrained("llama3_1_tax_lora")
tokenizer.save_pretrained("llama3_1_tax_lora")
print("Training complete! Model saved in 'llama3_1_tax_lora'")

# **Inference (Testing) Phase**

I used inference to demonstrate how my training improved the model. This is the "test the model" part of Task 2.

**Phase 1: Setting up the Generator**

I used a "pipeline," which is a high-level Hugging Face tool that handles the complex math of generating text so I don't have to write the loops yourself.

In [ ]:
import torch
from transformers import pipeline

# Setup the Generator using your fine-tuned model
# 'model' and 'tokenizer' should be the objects from my trainer
pipe = pipeline(task = "text-generation", model = model, tokenizer = tokenizer, max_new_tokens = 100)

**Phase 2: Defining the Test Case**

Here, I define the "unseen" question. This is the moment of truth to see if the fine-tuning actually worked.

In [ ]:
# Pick a specific tax question from dataset_clean.csv I want the model to answer
unseen_question = "Wie hoch ist der Alleinverdienerabsetzbetrag für ein Kind ab 2025?"

**Phase 3: Formatting the Input**

**CRITICAL STEP:** LLMs are sensitive to patterns. Since I trained the model using ### Frage: and ### Antwort:, I must present the test question in the exact same way, or the model might get confused and start babbling.

In [ ]:
# Format the prompt exactly like the training data
# This tells the model: "The training is over, now fill in the blank after 'Answer:'"
prompt = f"### Question:\n{unseen_question}\n\n### Answer:\n"

**Phase 4: Generation**

This is where the GPU processes the prompt and predicts the most likely next words based on the Austrian tax law it just learned.

In [ ]:
# Pass the formatted prompt to the pipeline to generate an answer
result = pipe(prompt)

# Extract the full text (which includes your prompt + the new answer)
generated_text = result[0]['generated_text']

**Phase 5: Cleaning and Display**

Because the model returns the entire conversation (Prompt + Answer), I "sliced" the text so you can only see the AI's specific answer.

In [ ]:
# Split the string at '### Answer:' and take the last part (the generated text)
# This removes the original question from the printout
# Print the result
print("TEST RESULT:")
print(generated_text.split("### Answer:\n")[-1])

# **(Bulk Inference) Production Phase**

While my previous script tested a single question, this code is designed to process the entire 644-row dataset as efficiently as possible by utilizing batching and multi-GPU power.

**Phase 1: Environment & Pipeline Setup**

I prepared the model to handle large volumes of data by spreading the workload across my available hardware.

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm
import csv

# Load the original test questions into a DataFrame
test_df = pd.read_csv('dataset_clean.csv')

# Initialize the generator pipeline
# This uses the fine-tuned 'model' and 'tokenizer' from the previous steps
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    device_map = "auto", # device_map = "auto" automatically distributes the model across the 4 GPUs
    dtype = torch.bfloat16 # bfloat16 is a data format that maintains high accuracy while using less memory
)

**Phase 2: Batching Strategy**

Processing questions one-by-one is slow. "Batching" sends multiple questions to the GPU simultaneously, which is like using an 8-lane highway instead of a single-lane road.

In [ ]:
# Setup batching parameters
batch_size = 8 # batch_size = 8 means we send 8 questions to the GPU at the exact same time
results = []

print(f"Starting inference for {len(test_df)} rows...")

# Run inference in batches
# Loop through the dataset in 'jumps' of 8 (0, 8, 16, 24...)
# tqdm creates a visual progress bar in my console
for i in tqdm(range(0, len(test_df), batch_size)):
    # Slice the DataFrame to get the current group of 8 rows
    batch = test_df.iloc[i:i+batch_size]

    # Format all 8 prompts exactly as the model was trained
    prompts = [f"### Question:\n{row['prompt']}\n\n### Answer:\n" for _, row in batch.iterrows()]

**Phase 3: The Generation Engine**

This is where the fine-tuned model does the heavy legal lifting.

In [ ]:
    # Generate responses
    # We set max_new_tokens to 150 to allow for detailed legal citations
    outputs = generator(
        prompts,
        max_new_tokens = 150,
        do_sample = False, # Set to False for deterministic, professional answers
        pad_token_id = tokenizer.eos_token_id,
        batch_size = batch_size
    )

**Phase 4: Data Cleaning & Extraction**

Since the model returns the prompt and the answer together, I needed to strip away the prompt to leave only the professional legal response.

In [ ]:
    # Clean and store the outputs
    for j, output in enumerate(outputs): # Loop through the 8 outputs generated in this batch
        generated_text = output[0]['generated_text']
        # Slices the text to keep only what comes AFTER the "### Answer:" marker
        clean_answer = generated_text.split("### Answer:\n")[-1].strip()

        # Map the clean answer back to its original ID for data integrity
        results.append({
            "id": batch.iloc[j]['id'],
            "answer": clean_answer
        })

**Phase 5: Secure Export**

Finally, I saved the results. Since tax law is full of specific punctuation, I used strict CSV settings to prevent the file from becoming corrupted.

In [ ]:
# Define the output filename
output_file = 'fine_tuned_results.csv'
results_df = pd.DataFrame(results)

# quoting = csv.QUOTE_ALL wraps every single cell in "quotes"
# This prevents a comma in a law citation (e.g., "§ 2, Abs. 1") from being mistaken for a new column
results_df.to_csv(output_file, index = False, quoting = csv.QUOTE_ALL, encoding = 'utf-8')

print(f"\nSuccess! Inference complete. File saved as: {output_file}")

#### **Summary**

**Baseline:** I used a general model (GPT-4o-mini) to see how it performed.

**Training:** I took a specialized model (Llama-3.1-8B) and "taught" it Austrian Tax Law using Unsloth.

**Validation:** I tested it with a single "unseen" question.

**Production:** I used this script to generate high-quality, professional answers for your entire dataset using high-performance batching.

While the model achieved a 0.225 training loss and correctly identifies legal structures (§ 33 EStG), it currently uses 2022/2023 values for indexation. For production, Task 3 (RAG) would be integrated to fetch live 2025 tables.